# Portfolio Optimization — ทุก Steps ด้วย Linear Algebra (SET Data จริง)

Notebook นี้ใช้ข้อมูลจริงจาก:
- **`close_set.csv`** / **`net_profit_set.csv`** — กำไรสุทธิรายไตรมาส (forward-filled รายวัน) ของหุ้น SET 825 ตัว ปี 2015–2025

ครอบคลุมทุก 7 หัวข้อจาก `explanation.md`:

| Step | หัวข้อ | เทคนิค |
|:----:|:-------|:-------|
| 1 | Portfolio Variance & Markowitz | Matrix inversion, Quadratic Form, QP |
| 2 | PCA on Covariance Matrix | Eigen-decomposition Σ=VΛVᵀ |
| 3 | Factor Models & OLS | Normal Equation, QR decomposition |
| 4 | Cointegration & Kalman Filter | State-space, dynamic hedge ratio |
| 5 | Random Matrix Theory | Marchenko-Pastur, Ledoit-Wolf |
| 6 | VaR & Cholesky | Σ=LLᵀ, Monte Carlo |
| 7 | Black-Litterman | Bayesian updating, matrix inversion |

## ⚙️ Setup

In [ ]:
!pip install numpy scipy matplotlib pandas scikit-learn statsmodels --quiet

## Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from scipy.optimize import minimize
from sklearn.covariance import LedoitWolf
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint, adfuller

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
print('Import OK')

## Data Loading — SET Net Profit Data

ทั้ง `close_set.csv` และ `net_profit_set.csv` เก็บ **กำไรสุทธิรายไตรมาส** (quarterly net profit) ที่ถูก forward-fill รายวัน ค่าจึงเปลี่ยนทุก ~60 วัน (ทุกไตรมาส)

### วิธีประมวลผลที่ถูกต้อง
1. Resample ไปเป็น **Quarterly** snapshot (QE) ก่อน
2. คำนวณ **QoQ change** (quarter-over-quarter) → ใช้เป็น "return" สำหรับ portfolio analysis
3. คัดหุ้นที่มีข้อมูลครบและมี variance สูง (active, informative)
4. ใช้ **n=15 หุ้น** จาก T=41 ไตรมาส → T >> n → Σ full rank และ invertible

In [ ]:
# ============================================================
# DATA LOADING & PREPARATION
# ============================================================

print('Loading close_set.csv ...')
raw = pd.read_csv('close_set.csv', index_col=0, parse_dates=True)
print(f'Raw shape: {raw.shape}  ({raw.index[0].date()} to {raw.index[-1].date()})')

# ---- Step 1: Resample to Quarterly ----
# Each quarter uses the last observation (most recent net profit reported)
net_q = raw.resample('QE').last()
print(f'Quarterly shape: {net_q.shape}  ({net_q.shape[0]} quarters)')

# ---- Step 2: Select stocks with ZERO missing quarters ----
null_counts = net_q.isnull().sum()
perfect_stocks = null_counts[null_counts == 0].index.tolist()
print(f'Stocks with complete quarterly data: {len(perfect_stocks)}')

# ---- Step 3: Compute Quarter-over-Quarter returns ----
ret_q_all = (
    net_q[perfect_stocks]
    .pct_change(1, fill_method=None)
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .clip(-5, 5)     # cap outliers at +/-500% QoQ change
)
print(f'Quarterly returns shape: {ret_q_all.shape}  (T={ret_q_all.shape[0]} quarters)')

# ---- Step 4: Select top-15 most variable stocks ----
# More variance = more informative earnings signal
# Also ensures Sigma is well-conditioned (T=41 >> n=15)
TOP_N = 15
var_rank = ret_q_all.var().sort_values(ascending=False)
tickers  = var_rank.head(TOP_N).index.tolist()
n = len(tickers)

returns = ret_q_all[tickers].copy()
T = len(returns)

print(f'\nSelected {n} stocks (highest QoQ variance, T={T} quarters):')
print(tickers)
print(f'Date range: {returns.index[0].date()} to {returns.index[-1].date()}')

# ---- Verify Sigma is invertible ----
R     = returns.values                 # T x n matrix
Sigma = np.cov(R.T)                   # n x n covariance
evals = np.linalg.eigvalsh(Sigma)
cond  = np.linalg.cond(Sigma)
print(f'\nCovariance matrix check:')
print(f'  Rank: {np.linalg.matrix_rank(Sigma)}/{n}')
print(f'  Min eigenvalue: {evals.min():.4f}  (>0 = positive definite)')
print(f'  Condition number: {cond:.2e}  (well-conditioned if < 1e6)')

# ---- Also prepare YoY growth for factor/BL views ----
yoy_q = (
    net_q[tickers]
    .pct_change(4, fill_method=None)
    .replace([np.inf, -np.inf], np.nan)
    .clip(-5, 5)
)
print(f'\nYoY growth shape: {yoy_q.shape}')
print('\nSample quarterly returns (last 4 rows, first 5 stocks):')
returns.iloc[-4:, :5].round(4)

---
# Step 1: Portfolio Variance & Markowitz Optimization

## หลักการ (explanation.md §1)
- Portfolio return: $R_p = w^\top r$, expected: $E[R_p] = w^\top \mu$
- Portfolio variance (quadratic form): $\text{Var}(R_p) = w^\top \Sigma w$

## Minimum Variance Portfolio
$$\min_w \; w^\top \Sigma w \quad \text{s.t.} \quad w^\top \mathbf{1} = 1$$
Lagrangian: $L = w^\top\Sigma w - \lambda(w^\top\mathbf{1}-1)$ → $w^* = \dfrac{\Sigma^{-1}\mathbf{1}}{\mathbf{1}^\top\Sigma^{-1}\mathbf{1}}$

In [ ]:
# ============================================================
# STEP 1: Markowitz Portfolio Optimization
# Data: quarterly QoQ net-profit changes (T=41, n=15 SET stocks)
# ============================================================

mu    = returns.mean().values     # quarterly expected return
ones  = np.ones(n)

# ---- 1a. Closed-form Minimum Variance Portfolio ----
# w* = Sigma^{-1} 1 / (1^T Sigma^{-1} 1)
Sigma_inv = np.linalg.inv(Sigma)
w_mvp     = Sigma_inv @ ones / (ones @ Sigma_inv @ ones)

ret_mvp  = mu @ w_mvp
risk_mvp = np.sqrt(w_mvp @ Sigma @ w_mvp)

print("=" * 65)
print("Minimum Variance Portfolio (Closed-form) — SET Net-Profit Data")
print("=" * 65)
print(f"{'Ticker':>8} | {'Weight':>8} | {'mu (qtr)':>10} | {'sigma (qtr)':>12}")
print("-" * 55)
for t, w, m, s in zip(tickers, w_mvp, mu, np.sqrt(np.diag(Sigma))):
    print(f"{t:>8} | {w*100:>7.2f}% | {m*100:>9.2f}% | {s*100:>11.2f}%")
print(f"\nMVP -> Return: {ret_mvp*100:.2f}%/qtr, sigma: {risk_mvp*100:.2f}%/qtr, Sharpe: {ret_mvp/risk_mvp:.3f}")

# ---- Helper functions ----
def portfolio_variance(w, Sigma):
    return w @ Sigma @ w

def portfolio_return(w, mu):
    return w @ mu

def neg_sharpe(w, mu, Sigma, rf=0.005):   # rf = 0.5%/quarter (~2%/yr)
    ret = mu @ w
    vol = np.sqrt(w @ Sigma @ w)
    return -(ret - rf) / vol

# ---- 1b. Efficient Frontier (No Short-Selling, Quadratic Programming) ----
target_returns = np.linspace(mu.min(), mu.max(), 60)
frontier_risks, frontier_rets = [], []

for target in target_returns:
    constraints = [
        {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
        {'type': 'eq', 'fun': lambda w, t=target: portfolio_return(w, mu) - t}
    ]
    result = minimize(
        portfolio_variance, x0=ones/n,
        args=(Sigma,), method='SLSQP',
        bounds=[(0, 1)] * n, constraints=constraints,
        options={'ftol': 1e-10, 'maxiter': 1000}
    )
    if result.success:
        frontier_risks.append(np.sqrt(result.fun))
        frontier_rets.append(target)

# ---- Max Sharpe Ratio Portfolio ----
result_sr = minimize(
    neg_sharpe, x0=ones/n,
    args=(mu, Sigma), method='SLSQP',
    bounds=[(0, 1)] * n,
    constraints=[{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}],
    options={'ftol': 1e-10, 'maxiter': 1000}
)
w_sr    = result_sr.x
ret_sr  = mu @ w_sr
risk_sr = np.sqrt(w_sr @ Sigma @ w_sr)
print(f"Max Sharpe -> Return: {ret_sr*100:.2f}%/qtr, sigma: {risk_sr*100:.2f}%/qtr, Sharpe: {ret_sr/risk_sr:.3f}")

# ---- Plot ----
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
ax.plot(np.array(frontier_risks)*100, np.array(frontier_rets)*100,
        'b-', lw=2.5, label='Efficient Frontier (No Short)')
ax.scatter(risk_mvp*100, ret_mvp*100, s=200, color='green',
           zorder=5, marker='*', label=f'Min Variance')
ax.scatter(risk_sr*100, ret_sr*100, s=200, color='red',
           zorder=5, marker='D', label=f'Max Sharpe ({ret_sr/risk_sr:.2f})')
for i, t in enumerate(tickers):
    ax.scatter(np.sqrt(Sigma[i,i])*100, mu[i]*100, s=80, alpha=0.7, zorder=4)
    ax.annotate(t, (np.sqrt(Sigma[i,i])*100, mu[i]*100),
                textcoords='offset points', xytext=(4, 2), fontsize=8)
ax.set_xlabel('Volatility (sigma) % per quarter')
ax.set_ylabel('Expected Return % per quarter')
ax.set_title('Efficient Frontier — SET Net-Profit Quarterly Returns')
ax.legend()

ax2 = axes[1]
colors = plt.cm.tab20(np.linspace(0, 1, n))
ax2.bar(range(n), w_sr * 100, color=colors)
ax2.set_xticks(range(n))
ax2.set_xticklabels(tickers, rotation=45, fontsize=9)
ax2.set_ylabel('Weight (%)')
ax2.set_title(f'Max Sharpe Portfolio Weights\nReturn={ret_sr*100:.1f}%, sigma={risk_sr*100:.1f}%, Sharpe={ret_sr/risk_sr:.2f}')
ax2.axhline(0, color='k', lw=0.8)

plt.tight_layout()
plt.savefig('step1_markowitz.png', dpi=120, bbox_inches='tight')
plt.show()
print('Step 1 done')

---
# Step 2: PCA on Covariance Matrix

## หลักการ (explanation.md §2)
Sigma เป็น symmetric matrix -> diagonalize ได้:
$$\Sigma = V \Lambda V^\top$$
- $V$ = eigenvectors (orthonormal columns)
- $\Lambda$ = diagonal eigenvalues (descending)
- **% variance explained** by factor $i$ = $\lambda_i / \sum_j\lambda_j$
- PC1 มักเป็น "market factor" (Perron-Frobenius: loading บวกทุกตัว)

In [ ]:
# ============================================================
# STEP 2: PCA on Covariance Matrix (Sigma = V Lambda V^T)
# ============================================================

R_demean = R - R.mean(axis=0)   # zero-mean returns (T x n)

# Eigen-decomposition of sample covariance matrix
eigenvalues, evecs = np.linalg.eigh(Sigma)   # eigh for symmetric
idx          = np.argsort(eigenvalues)[::-1]  # sort descending
eigenvalues  = eigenvalues[idx]
eigenvectors = evecs[:, idx]

explained_var  = eigenvalues / eigenvalues.sum() * 100
cumulative_var = np.cumsum(explained_var)

print("Eigenvalue Analysis — SET Net-Profit Quarterly Returns")
print(f"{'PC':>4} | {'Eigenvalue':>12} | {'Explained%':>11} | {'Cumulative%':>12}")
print("-" * 50)
for i in range(n):
    marker = " <- market/earnings factor" if i == 0 else ""
    print(f"  {i+1:2d} | {eigenvalues[i]:12.4f} | {explained_var[i]:10.2f}% | {cumulative_var[i]:11.2f}%{marker}")

# Project onto top-k factors
k          = 3
V_k        = eigenvectors[:, :k]   # n x k
factor_ret = R_demean @ V_k        # T x k  (factor time series)

# Residuals = idiosyncratic component
residuals = R_demean - factor_ret @ V_k.T   # T x n

print(f"\nPC1 explains {explained_var[0]:.1f}% of variance")
print(f"PC1-3 explain {cumulative_var[2]:.1f}% of variance")

# ---- Plot ----
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Scree plot
ax = axes[0, 0]
ax.bar(range(1, n+1), explained_var, color='steelblue', alpha=0.7)
ax2t = ax.twinx()
ax2t.plot(range(1, n+1), cumulative_var, 'r-o', ms=5)
ax2t.set_ylabel('Cumulative % Explained', color='red')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Variance Explained (%)')
ax.set_title('Scree Plot — SET Net-Profit Returns')
ax.axvline(k + 0.5, color='orange', ls='--', lw=2, label=f'Top {k} PCs')
ax.legend()

# 2. Factor Loadings Heatmap
ax = axes[0, 1]
im = ax.imshow(V_k.T, aspect='auto', cmap='RdBu_r', vmin=-0.7, vmax=0.7)
ax.set_xticks(range(n))
ax.set_xticklabels(tickers, rotation=45, fontsize=8)
ax.set_yticks(range(k))
ax.set_yticklabels([f'PC{i+1}' for i in range(k)])
ax.set_title(f'Factor Loadings (Top {k} PCs) — eigenvectors')
plt.colorbar(im, ax=ax)
for i in range(k):
    for j in range(n):
        ax.text(j, i, f'{V_k[j,i]:.2f}', ha='center', va='center', fontsize=7)

# 3. PC1 (Market/Earnings Factor) Time-Series
ax = axes[1, 0]
ax.bar(range(T), factor_ret[:, 0], color=['green' if v > 0 else 'red' for v in factor_ret[:, 0]], alpha=0.7)
ax.set_title('PC1 (Primary Factor) — Quarterly')
ax.set_xlabel('Quarter index')
ax.set_ylabel('Factor Return')

# 4. Idiosyncratic Residual of first stock
ax = axes[1, 1]
res0 = residuals[:, 0]
ax.bar(range(T), res0, color=['green' if v > 0 else 'red' for v in res0], alpha=0.7)
ax.axhline(res0.mean(), color='black', ls='--', lw=1.5, label='Mean')
ax.set_title(f'Idiosyncratic Residual — {tickers[0]}')
ax.set_xlabel('Quarter index')
ax.legend()

plt.tight_layout()
plt.savefig('step2_pca.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Step 2 done — PC1-3 explain {cumulative_var[2]:.1f}% of variance')

---
# Step 3: Factor Models & OLS

## OLS — Normal Equation (explanation.md §3)
$$\hat{\beta} = (X^\top X)^{-1} X^\top r$$

ในทางปฏิบัติใช้ **QR decomposition** แทน (เร็วกว่าและ numerically stable):
$$X = QR \Rightarrow \hat{\beta} = R^{-1} Q^\top r$$

## Factors
- **Factor 1** (Market/Earnings): PC1 จาก Step 2
- **Factor 2** (Size/Sector): PC2
- **Factor 3** (Momentum): PC3

In [ ]:
# ============================================================
# STEP 3: Factor Models & OLS
# Factors: PC1, PC2, PC3 from Step 2
# ============================================================

# Design matrix: intercept + 3 PCA factors
X = np.hstack([np.ones((T, 1)), factor_ret])   # T x 4
Y = R_demean                                    # T x n

# ---- Method 1: Direct Normal Equation ----
B_direct = np.linalg.inv(X.T @ X) @ X.T @ Y   # 4 x n

# ---- Method 2: QR Decomposition (numerically stable) ----
# X = QR  ->  beta = R^{-1} Q^T r
Q_qr, R_qr = np.linalg.qr(X)
B_qr       = np.linalg.solve(R_qr, Q_qr.T @ Y)

max_diff = np.max(np.abs(B_direct - B_qr))
print(f"Max |B_direct - B_QR| = {max_diff:.2e}  (should be ~0)")

# ---- Method 3: statsmodels (t-stat, p-value, R^2) ----
print("\n" + "=" * 70)
print("OLS Regression: r_i = alpha + beta1*PC1 + beta2*PC2 + beta3*PC3")
print("=" * 70)
print(f"{'Ticker':>8} | {'alpha(%/qtr)':>13} | {'b_PC1':>7} | {'b_PC2':>7} | {'b_PC3':>7} | {'R2':>6}")
print("-" * 62)

results_ols = {}
for i, ticker in enumerate(tickers):
    model = sm.OLS(Y[:, i], X).fit()
    results_ols[ticker] = model
    alpha = model.params[0] * 100
    betas = model.params[1:]
    r2    = model.rsquared
    print(f"{ticker:>8} | {alpha:>13.3f} | {betas[0]:>7.3f} | {betas[1]:>7.3f} | {betas[2]:>7.3f} | {r2:>6.3f}")

# ---- Plot ----
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Beta heatmap (PC1-3)
B_plot = B_direct[1:4, :]   # 3 x n
ax = axes[0]
im = ax.imshow(B_plot, aspect='auto', cmap='coolwarm', vmin=-2, vmax=2)
ax.set_yticks(range(3))
ax.set_yticklabels(['beta_PC1', 'beta_PC2', 'beta_PC3'])
ax.set_xticks(range(n))
ax.set_xticklabels(tickers, rotation=45, fontsize=8)
ax.set_title('Factor Beta Heatmap (OLS) — QoQ Net-Profit Returns')
plt.colorbar(im, ax=ax)
for row in range(3):
    for col in range(n):
        ax.text(col, row, f'{B_plot[row,col]:.2f}', ha='center', va='center', fontsize=7)

# R^2 bar chart
r2_vals = [results_ols[t].rsquared for t in tickers]
ax2 = axes[1]
bars = ax2.bar(range(n), [v*100 for v in r2_vals],
               color=plt.cm.viridis(np.array(r2_vals)))
ax2.set_xticks(range(n))
ax2.set_xticklabels(tickers, rotation=45, fontsize=8)
ax2.set_ylabel('R^2 (%)')
ax2.set_title('R^2 per Stock (3-Factor PCA Model)')
ax2.set_ylim(0, 100)
for bar, val in zip(bars, r2_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val*100:.0f}%', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('step3_ols.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Step 3 done — Avg R^2: {np.mean(r2_vals)*100:.1f}%')

---
# Step 4: Cointegration, Kalman Filter & Pairs Trading

## หลักการ (explanation.md §4)
- **Correlation สูง != เทรดได้** — ราคา non-stationary อาจ diverge (spurious)
- **Cointegration**: หา linear combination ของ series 2 ตัวที่ stationary (mean-reverting)

## Kalman Filter — Dynamic Hedge Ratio
$$\text{State: } \beta_t = \beta_{t-1} + w_t \quad \text{Obs: } y_{1t} = \beta_t y_{2t} + v_t$$
- Predict: $P_{t|t-1} = P_{t-1} + Q$
- Kalman Gain: $K_t = P_{t|t-1}H^\top(HP_{t|t-1}H^\top + R)^{-1}$
- Update: $\hat{\beta}_t = \hat{\beta}_{t-1} + K_t(y_{1t} - H\hat{\beta}_{t-1})$

In [ ]:
# ============================================================
# STEP 4: Cointegration & Kalman Filter
# Data: quarterly net-profit levels (not returns)
# ============================================================

# Use quarterly net-profit LEVELS for cointegration test
# (cointegration tests run on non-stationary levels, not returns)
net_levels = net_q[tickers].dropna()
prices = net_levels   # shape: Q x n (quarterly levels)

print("Testing Cointegration for all pairs (Engle-Granger)...")
pairs_results = []
for i in range(n):
    for j in range(i+1, n):
        t1, t2 = tickers[i], tickers[j]
        s1 = prices[t1].values.astype(float)
        s2 = prices[t2].values.astype(float)
        try:
            score, pvalue, _ = coint(s1, s2)
            pairs_results.append((t1, t2, pvalue, score))
        except:
            pass

pairs_df = pd.DataFrame(pairs_results, columns=['Ticker1','Ticker2','p-value','Score'])
pairs_df = pairs_df.sort_values('p-value').reset_index(drop=True)

print("\nTop 5 Cointegrated Pairs (SET net-profit levels):")
print(pairs_df.head(5).to_string(index=False))

best = pairs_df.iloc[0]
tk1, tk2 = best['Ticker1'], best['Ticker2']
print(f"\nBest pair: {tk1} - {tk2}  (p={best['p-value']:.4f})")

y1 = prices[tk1].values.astype(float)
y2 = prices[tk2].values.astype(float)

# ---- Kalman Filter for Dynamic Hedge Ratio ----
def kalman_filter_hedge(y1, y2, delta=1e-4, vt=1e-2):
    """Kalman Filter: dynamic hedge ratio beta_t"""
    T_kf   = len(y1)
    beta   = np.zeros(T_kf)
    P      = np.zeros(T_kf)
    spread = np.zeros(T_kf)

    beta[0] = y1[0] / (y2[0] + 1e-10)
    P[0]    = 1.0
    Q_noise = delta / (1 - delta)   # process noise
    R_noise = vt                    # observation noise

    for t in range(1, T_kf):
        # Predict (F=1, state = random walk)
        beta_pred = beta[t-1]
        P_pred    = P[t-1] + Q_noise

        # Kalman Gain: K = P H^T (H P H^T + R)^{-1}
        H  = y2[t]
        S  = H * P_pred * H + R_noise
        Kg = P_pred * H / S

        # Update
        innovation = y1[t] - H * beta_pred
        beta[t]    = beta_pred + Kg * innovation
        P[t]       = (1 - Kg * H) * P_pred
        spread[t]  = y1[t] - beta[t] * y2[t]

    return beta, spread

beta_kf, spread_kf = kalman_filter_hedge(y1, y2)

# Static OLS hedge ratio
X_ols     = sm.add_constant(y2)
ols_model = sm.OLS(y1, X_ols).fit()
beta_ols  = ols_model.params[1]

# Z-score (rolling 6-quarter window)
window     = 6
s_ser      = pd.Series(spread_kf)
s_mean     = s_ser.rolling(window).mean()
s_std      = s_ser.rolling(window).std()
zscore     = (s_ser - s_mean) / (s_std + 1e-10)

# ADF test on spread
adf_result = adfuller(spread_kf[window:], autolag='AIC')
print(f"\nADF Test on Kalman spread: stat={adf_result[0]:.4f}, p={adf_result[1]:.4f}")
print("  Spread is", "stationary" if adf_result[1] < 0.05 else "NOT stationary")

# ---- Plot ----
fig, axes = plt.subplots(3, 1, figsize=(14, 12))
q_idx = prices.index

ax = axes[0]
ax.plot(q_idx, y1/y1[0]*100, label=tk1, lw=1.5)
ax.plot(q_idx, y2/y2[0]*100, label=tk2, lw=1.5)
ax.set_title(f'Net-Profit Levels: {tk1} vs {tk2} (rebased=100)')
ax.set_ylabel('Net Profit (rebased=100)')
ax.legend()

ax = axes[1]
ax.plot(q_idx, beta_kf, label='Kalman (dynamic)', color='navy', lw=1.5)
ax.axhline(beta_ols, color='red', ls='--', lw=1.5, label=f'OLS static beta={beta_ols:.4f}')
ax.set_title('Hedge Ratio Over Time (Quarterly)')
ax.set_ylabel('Hedge Ratio beta')
ax.legend()

ax = axes[2]
ax.plot(q_idx, zscore, lw=1.2, color='darkgreen')
ax.axhline( 2, color='red',   ls='--', lw=1, label='±2sigma signal')
ax.axhline(-2, color='red',   ls='--', lw=1)
ax.axhline( 0, color='black', ls='-',  lw=0.8)
ax.fill_between(q_idx,  2, zscore.values, where=(zscore.values > 2),
                alpha=0.3, color='red',   label='Short spread')
ax.fill_between(q_idx, -2, zscore.values, where=(zscore.values < -2),
                alpha=0.3, color='green', label='Long spread')
ax.set_title('Z-Score of Spread (Trading Signal)')
ax.set_ylabel('Z-Score')
ax.legend()

plt.tight_layout()
plt.savefig('step4_kalman.png', dpi=120, bbox_inches='tight')
plt.show()
print('Step 4 done')

---
# Step 5: Random Matrix Theory & Covariance Denoising

## หลักการ (explanation.md §5)
- ถ้า $n/T$ ไม่เล็กมาก eigenvalue เล็กๆ จะถูกประเมินผิดพลาดสูง
- **Marchenko-Pastur bounds**: eigenvalue ของ pure noise จะอยู่ใน $[\lambda_{min}, \lambda_{max}]$

$$\lambda_{max} = \sigma^2\left(1+\sqrt{1/q}\right)^2, \quad q = T/n$$

## วิธีแก้
1. **Eigenvalue Clipping**: $\Sigma_{clean} = V\Lambda_{clip}V^\top$
2. **Ledoit-Wolf Shrinkage**: $\Sigma_{shrunk} = \delta F + (1-\delta)\Sigma_{sample}$

In [ ]:
# ============================================================
# STEP 5: Random Matrix Theory & Covariance Denoising
# ============================================================

T_obs, n_assets = R.shape
q = T_obs / n_assets

# Sample covariance
Sigma_s = np.cov(R.T)

# Eigen-decomposition: Sigma = V Lambda V^T
evals_s, evecs_s = np.linalg.eigh(Sigma_s)
idx_s    = np.argsort(evals_s)[::-1]
evals_s  = evals_s[idx_s]
evecs_s  = evecs_s[:, idx_s]

# Marchenko-Pastur bounds
var_scale       = np.mean(np.diag(Sigma_s))
lam_max_scaled  = var_scale * (1 + np.sqrt(1/q))**2
lam_min_scaled  = var_scale * (1 - np.sqrt(1/q))**2

signal_mask = evals_s > lam_max_scaled
noise_mask  = ~signal_mask
n_signal    = signal_mask.sum()

print(f"q = T/n = {T_obs}/{n_assets} = {q:.2f}")
print(f"Marchenko-Pastur lambda_max = {lam_max_scaled:.4f}")
print(f"Marchenko-Pastur lambda_min = {lam_min_scaled:.4f}")
print(f"\nSignal eigenvalues (> lambda_max): {n_signal}  | Noise: {noise_mask.sum()}")

# ---- Method 1: Eigenvalue Clipping ----
evals_clipped             = evals_s.copy()
noise_mean                = evals_s[noise_mask].mean() if noise_mask.any() else 0
evals_clipped[noise_mask] = noise_mean
Sigma_clipped = evecs_s @ np.diag(evals_clipped) @ evecs_s.T

# ---- Method 2: Ledoit-Wolf Shrinkage ----
lw              = LedoitWolf().fit(R)
Sigma_lw        = lw.covariance_
shrinkage_coeff = lw.shrinkage_
print(f"Ledoit-Wolf shrinkage delta = {shrinkage_coeff:.4f}")

# Condition numbers
cond_sample  = np.linalg.cond(Sigma_s)
cond_clipped = np.linalg.cond(Sigma_clipped)
cond_lw      = np.linalg.cond(Sigma_lw)
print(f"\nCondition Number Comparison:")
print(f"  Sample Covariance   : {cond_sample:12.1f}")
print(f"  Eigenvalue Clipping : {cond_clipped:12.1f}")
print(f"  Ledoit-Wolf         : {cond_lw:12.1f}")

# Marchenko-Pastur PDF
def marchenko_pastur_pdf(x, q, sigma2=1.0):
    lmax = sigma2 * (1 + np.sqrt(1/q))**2
    lmin = sigma2 * (1 - np.sqrt(1/q))**2
    mask = (x >= lmin) & (x <= lmax)
    pdf  = np.zeros_like(x)
    pdf[mask] = (q / (2 * np.pi * sigma2 * x[mask])) * \
                np.sqrt((lmax - x[mask]) * (x[mask] - lmin))
    return pdf

# ---- Plot ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Eigenvalue distribution vs MP
ax = axes[0]
noise_evals = evals_s[noise_mask] if noise_mask.any() else evals_s
ax.hist(evals_s, bins=10, density=True, alpha=0.6, color='steelblue',
        label='Sample eigenvalues')
x_range = np.linspace(max(0.01, lam_min_scaled*0.5), lam_max_scaled*1.5, 300)
ax.plot(x_range, marchenko_pastur_pdf(x_range, q, sigma2=var_scale),
        'r-', lw=2, label='Marchenko-Pastur PDF')
ax.axvline(lam_max_scaled, color='orange', ls='--', lw=2, label=f'lambda_max={lam_max_scaled:.3f}')
ax.set_xlabel('Eigenvalue')
ax.set_ylabel('Density')
ax.set_title('Eigenvalue Dist. vs Marchenko-Pastur\n(SET Net-Profit, q=T/n={:.1f})'.format(q))
ax.legend(fontsize=8)

# 2. Std dev comparison
ax = axes[1]
diag_s  = np.sqrt(np.diag(Sigma_s))
diag_c  = np.sqrt(np.diag(Sigma_clipped))
diag_lw = np.sqrt(np.diag(Sigma_lw))
x_ = np.arange(n_assets)
w_ = 0.28
ax.bar(x_ - w_, diag_s,  w_, label='Sample',      color='steelblue',  alpha=0.8)
ax.bar(x_,      diag_c,  w_, label='Clipped',     color='darkorange', alpha=0.8)
ax.bar(x_ + w_, diag_lw, w_, label='Ledoit-Wolf', color='green',      alpha=0.8)
ax.set_xticks(x_)
ax.set_xticklabels(tickers, rotation=45, fontsize=7)
ax.set_ylabel('Std Dev (quarterly)')
ax.set_title('Diagonal of Covariance — 3 Methods')
ax.legend()

# 3. Ledoit-Wolf Correlation Heatmap
ax = axes[2]
D_inv   = np.diag(1.0 / (np.sqrt(np.diag(Sigma_lw)) + 1e-10))
corr_lw = D_inv @ Sigma_lw @ D_inv
im = ax.imshow(corr_lw, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(n_assets))
ax.set_yticks(range(n_assets))
ax.set_xticklabels(tickers, rotation=45, fontsize=7)
ax.set_yticklabels(tickers, fontsize=7)
ax.set_title('Ledoit-Wolf Correlation Matrix (SET)')
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig('step5_rmt.png', dpi=120, bbox_inches='tight')
plt.show()
print('Step 5 done')

---
# Step 6: VaR & Cholesky Decomposition

## หลักการ (explanation.md §6)
$$\text{VaR}_{\alpha} = z_\alpha \cdot \sqrt{w^\top\Sigma w} \cdot \text{Portfolio Value}$$

## Monte Carlo ด้วย Cholesky
1. Decompose $\Sigma = LL^\top$ (L = lower triangular)
2. สุ่ม $z \sim N(0, I)$
3. Correlated returns: $r = \mu + Lz$

**พิสูจน์**: $\text{Cov}(Lz) = L \cdot I \cdot L^\top = LL^\top = \Sigma$ ✓

In [ ]:
# ============================================================
# STEP 6: VaR & Cholesky Decomposition
# Portfolio: Max Sharpe weights (Step 1)
# Covariance: Ledoit-Wolf (Step 5)
# ============================================================

portfolio_value  = 10_000_000   # 10 million baht
confidence_level = 0.99

w = w_sr   # Max Sharpe weights

# Ledoit-Wolf quarterly covariance
Sigma_q   = Sigma_lw.copy()
mu_q      = mu.copy()

# ---- Method 1: Variance-Covariance VaR (Analytical) ----
port_vol = np.sqrt(w @ Sigma_q @ w)
z_score  = stats.norm.ppf(confidence_level)
VaR_vc   = z_score * port_vol * portfolio_value
CVaR_vc  = (stats.norm.pdf(z_score) / (1 - confidence_level)) * port_vol * portfolio_value

print(f"Portfolio Volatility (quarterly): {port_vol*100:.4f}%")
print(f"\n{'='*55}")
print(f"Variance-Covariance VaR@99%:  {VaR_vc:>15,.0f} baht")
print(f"Variance-Covariance CVaR@99%: {CVaR_vc:>15,.0f} baht")

# ---- Method 2: Monte Carlo with Cholesky ----
# Sigma = L L^T  (Cholesky decomposition)
n_sim = 100_000
L = np.linalg.cholesky(Sigma_q)              # n x n lower triangular

Z             = np.random.standard_normal((n_assets, n_sim))   # n x N
corr_returns  = (mu_q[:, None] + L @ Z).T                      # N x n
portfolio_pnl = corr_returns @ w * portfolio_value              # N

VaR_mc   = -np.percentile(portfolio_pnl, (1 - confidence_level) * 100)
CVaR_mc  = -portfolio_pnl[portfolio_pnl < -VaR_mc].mean()

print(f"Monte Carlo VaR@99%  ({n_sim:,} sims): {VaR_mc:>12,.0f} baht")
print(f"Monte Carlo CVaR@99% ({n_sim:,} sims): {CVaR_mc:>12,.0f} baht")

# ---- Method 3: Historical VaR ----
hist_pnl  = (R @ w) * portfolio_value
VaR_hist  = -np.percentile(hist_pnl, (1 - confidence_level) * 100)
CVaR_hist = -hist_pnl[hist_pnl < -VaR_hist].mean()
print(f"Historical VaR@99%:                   {VaR_hist:>12,.0f} baht")
print(f"Historical CVaR@99%:                  {CVaR_hist:>12,.0f} baht")

# Verify Cholesky: L L^T = Sigma
recon_err = np.max(np.abs(L @ L.T - Sigma_q))
print(f"\nMax |LL^T - Sigma| = {recon_err:.2e}  (should be ~0)")

# ---- Plot ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.hist(portfolio_pnl / 1e6, bins=200, density=True,
        alpha=0.7, color='steelblue', label='MC Simulation')
ax.axvline(-VaR_mc/1e6,  color='red',    lw=2, ls='--',
           label=f'VaR@99%={VaR_mc/1e6:.2f}M')
ax.axvline(-CVaR_mc/1e6, color='darkred', lw=2, ls=':',
           label=f'CVaR={CVaR_mc/1e6:.2f}M')
ax.set_xlabel('Quarterly P&L (million baht)')
ax.set_ylabel('Density')
ax.set_title('Portfolio P&L Distribution (Monte Carlo)')
ax.legend(fontsize=9)

ax = axes[1]
im = ax.imshow(L, cmap='RdBu_r')
ax.set_xticks(range(n_assets))
ax.set_yticks(range(n_assets))
ax.set_xticklabels(tickers, rotation=45, fontsize=7)
ax.set_yticklabels(tickers, fontsize=7)
ax.set_title('Cholesky Factor L (Lower Triangular)')
plt.colorbar(im, ax=ax)

ax = axes[2]
methods   = ['Variance-\nCovariance', 'Monte Carlo\n(Cholesky)', 'Historical']
var_vals  = [VaR_vc/1e6, VaR_mc/1e6, VaR_hist/1e6]
cvar_vals = [CVaR_vc/1e6, CVaR_mc/1e6, CVaR_hist/1e6]
x_  = np.arange(len(methods))
ax.bar(x_ - 0.2, var_vals,  0.35, label='VaR@99%',  color='steelblue', alpha=0.85)
ax.bar(x_ + 0.2, cvar_vals, 0.35, label='CVaR@99%', color='darkorange', alpha=0.85)
ax.set_xticks(x_)
ax.set_xticklabels(methods)
ax.set_ylabel('Million baht')
ax.set_title('VaR & CVaR: 3 Methods Comparison')
ax.legend()

plt.tight_layout()
plt.savefig('step6_var_cholesky.png', dpi=120, bbox_inches='tight')
plt.show()
print('Step 6 done')

---
# Step 7: Black-Litterman Model

## หลักการ (explanation.md §7)

### ปัญหาของ Markowitz
- อ่อนไหวต่อ μ มาก — เปลี่ยน μ นิดเดียว weights เปลี่ยนสุดขั้ว

### Black-Litterman Solution
1. **Equilibrium Return**: $\Pi = \delta\Sigma w_{mkt}$
2. **Views**: P (view matrix), Q (view returns), Omega (view uncertainty)
3. **Bayesian Posterior**:
$$E[r] = \left[(\tau\Sigma)^{-1} + P^\top\Omega^{-1}P\right]^{-1} \left[(\tau\Sigma)^{-1}\Pi + P^\top\Omega^{-1}Q\right]$$

**Views จาก net_profit_set.csv**: หุ้นที่มี YoY earnings growth สูงสุด vs ต่ำสุด

In [ ]:
# ============================================================
# STEP 7: Black-Litterman Model
# Views: based on YoY net-profit growth from net_profit_set.csv
# ============================================================

delta = 2.5    # risk aversion
tau   = 0.05   # uncertainty in prior

# Market-cap weights (proxy: equal-weight)
w_mkt    = np.ones(n) / n
Sigma_ann = Sigma_lw.copy()

# ---- 7a. Equilibrium Return (Reverse Optimization) ----
# Pi = delta * Sigma * w_market
Pi = delta * Sigma_ann @ w_mkt

print("Equilibrium Returns Pi (implied by market weights):")
print(f"{'Ticker':>8} | {'Pi (quarterly)':>15}")
print("-" * 30)
for t, p in zip(tickers, Pi):
    print(f"{t:>8} | {p*100:>+15.3f}%")

# ---- 7b. Define Views from Net-Profit YoY growth ----
latest_yoy = yoy_q[tickers].dropna(how='all').iloc[-1].fillna(0)

# Rank stocks: positive YoY growth = expected outperformers
top2_idx    = latest_yoy.nlargest(2).index.tolist()
bottom2_idx = latest_yoy.nsmallest(2).index.tolist()

print(f"\nLatest YoY Net Profit Growth:")
print(f"  Top earners    : {top2_idx}  (YoY: {latest_yoy[top2_idx].values.round(3)})")
print(f"  Bottom earners : {bottom2_idx}  (YoY: {latest_yoy[bottom2_idx].values.round(3)})")

# P matrix (k_views x n)
k_views = 3
P = np.zeros((k_views, n))

# View 1: top earner[0] outperforms bottom earner[0] by 5% per quarter
P[0, tickers.index(top2_idx[0])]    =  1.0
P[0, tickers.index(bottom2_idx[0])] = -1.0

# View 2: top earner[1] outperforms bottom earner[1] by 3% per quarter
P[1, tickers.index(top2_idx[1])]    =  1.0
P[1, tickers.index(bottom2_idx[1])] = -1.0

# View 3: top2 average absolute return = +4% per quarter
for t in top2_idx:
    P[2, tickers.index(t)] = 0.5

Q     = np.array([0.05, 0.03, 0.04])  # view returns (quarterly)
Omega = np.diag(np.diag(tau * P @ Sigma_ann @ P.T))

print(f"\nViews (from net_profit_set.csv YoY):")
print(f"  View 1: {top2_idx[0]} outperforms {bottom2_idx[0]} by {Q[0]*100:.1f}%/qtr")
print(f"  View 2: {top2_idx[1]} outperforms {bottom2_idx[1]} by {Q[1]*100:.1f}%/qtr")
print(f"  View 3: avg({top2_idx}) = +{Q[2]*100:.1f}%/qtr")

# ---- 7c. Bayesian Posterior Return ----
# E[r] = [(tauSigma)^{-1} + P^T Omega^{-1} P]^{-1} [(tauSigma)^{-1} Pi + P^T Omega^{-1} Q]
tauSigma_inv = np.linalg.inv(tau * Sigma_ann)
Omega_inv    = np.linalg.inv(Omega)

M_post = np.linalg.inv(tauSigma_inv + P.T @ Omega_inv @ P)
mu_BL  = M_post @ (tauSigma_inv @ Pi + P.T @ Omega_inv @ Q)

print("\nBL Posterior vs Prior:")
print(f"{'Ticker':>8} | {'Prior Pi (%)':>12} | {'BL mu (%)':>10} | {'Delta (%)':>10}")
print("-" * 50)
for t, p, m in zip(tickers, Pi, mu_BL):
    print(f"{t:>8} | {p*100:>12.3f} | {m*100:>10.3f} | {(m-p)*100:>+10.3f}")

# ---- 7d. Optimize with BL vs Naive Returns ----
def optimize_portfolio(mu_in, Sigma_in):
    result = minimize(
        neg_sharpe, x0=np.ones(n)/n,
        args=(mu_in, Sigma_in),
        method='SLSQP',
        bounds=[(0, 1)] * n,
        constraints=[{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}],
        options={'ftol': 1e-12, 'maxiter': 2000}
    )
    return result.x

w_naive = optimize_portfolio(mu, Sigma_ann)
w_bl    = optimize_portfolio(mu_BL, Sigma_ann)

# Sensitivity: perturb mu by ±1% and check weight stability
n_perturb = 50
perturbation = 0.01
w_n_perturbed, w_b_perturbed = [], []
for _ in range(n_perturb):
    shock = np.random.normal(0, perturbation, n)
    w_n_perturbed.append(optimize_portfolio(mu + shock, Sigma_ann))
    w_b_perturbed.append(optimize_portfolio(mu_BL + shock, Sigma_ann))

std_naive = np.std(np.array(w_n_perturbed), axis=0)
std_bl    = np.std(np.array(w_b_perturbed), axis=0)
print(f"\nWeight Stability (avg std after +/-{perturbation*100:.1f}% shock):")
print(f"  Naive Markowitz : {std_naive.mean()*100:.2f}%")
print(f"  Black-Litterman : {std_bl.mean()*100:.2f}%")
if std_naive.mean() > 0:
    print(f"  BL reduces instability by {(1 - std_bl.mean()/std_naive.mean())*100:.0f}%")

# ---- Plot ----
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
x_ = np.arange(n)

ax = axes[0]
ax.bar(x_ - 0.2, Pi*100,    0.35, label='Prior Pi (Equilibrium)', color='steelblue', alpha=0.8)
ax.bar(x_ + 0.2, mu_BL*100, 0.35, label='BL Posterior mu',        color='darkorange', alpha=0.8)
ax.set_xticks(x_)
ax.set_xticklabels(tickers, rotation=45, fontsize=7)
ax.set_ylabel('Expected Return (% per quarter)')
ax.set_title('Prior vs Black-Litterman Return\n(Views from Net Profit YoY)')
ax.legend()

ax = axes[1]
ax.bar(x_ - 0.2, w_naive*100, 0.35, label='Naive Markowitz', color='steelblue', alpha=0.8)
ax.bar(x_ + 0.2, w_bl*100,    0.35, label='Black-Litterman', color='darkorange', alpha=0.8)
ax.set_xticks(x_)
ax.set_xticklabels(tickers, rotation=45, fontsize=7)
ax.set_ylabel('Weight (%)')
ax.set_title('Optimal Weights: Naive vs BL')
ax.legend()

ax = axes[2]
ax.bar(x_ - 0.2, std_naive*100, 0.35, label='Naive Markowitz', color='steelblue', alpha=0.8)
ax.bar(x_ + 0.2, std_bl*100,    0.35, label='Black-Litterman', color='darkorange', alpha=0.8)
ax.set_xticks(x_)
ax.set_xticklabels(tickers, rotation=45, fontsize=7)
ax.set_ylabel('Weight Std after Shock (%)')
ax.set_title(f'Weight Sensitivity ({n_perturb} perturbations, +/-{perturbation*100:.0f}%)')
ax.legend()

plt.tight_layout()
plt.savefig('step7_black_litterman.png', dpi=120, bbox_inches='tight')
plt.show()
print('Step 7 done')

---
# Final Summary — All 7 Steps

| Step | Topic | Linear Algebra Used | Data Source |
|:----:|:------|:--------------------|:------------|
| 1 | Markowitz | Matrix inv Sigma^{-1}, QP | close_set.csv (QoQ returns) |
| 2 | PCA | Eigen-decomp Sigma=V Lambda V^T | close_set.csv (QoQ returns) |
| 3 | OLS | Normal Eq, QR decomposition | QoQ returns + PC factors |
| 4 | Cointegration+Kalman | State-space, Kalman Gain matrix | close_set.csv (levels) |
| 5 | RMT | Marchenko-Pastur, Ledoit-Wolf | QoQ returns |
| 6 | VaR+Cholesky | Sigma=LL^T, quadratic form | QoQ returns + LW Sigma |
| 7 | Black-Litterman | Bayesian updating, matrix inv | net_profit_set.csv (YoY views) |

In [ ]:
# ============================================================
# FINAL SUMMARY
# ============================================================

summary_data = {
    'Step': ['1','2','3','4','5','6','7'],
    'Topic': [
        'Markowitz Optimization',
        'PCA on Covariance',
        'Factor Models & OLS',
        'Cointegration & Kalman Filter',
        'Random Matrix Theory',
        'VaR & Cholesky',
        'Black-Litterman'
    ],
    'Key Result': [
        f'Max Sharpe: Ret={ret_sr*100:.2f}%/qtr, sigma={risk_sr*100:.2f}%/qtr, Sharpe={ret_sr/risk_sr:.2f}',
        f'PC1-3 explain {cumulative_var[2]:.1f}% variance | PC1={explained_var[0]:.1f}%',
        f'Avg R2={np.mean([results_ols[t].rsquared for t in tickers])*100:.0f}% (3-factor PCA)',
        f'Best pair: {tk1}-{tk2} (p={best["p-value"]:.4f}) | ADF p={adf_result[1]:.4f}',
        f'Signal eigenvalues: {n_signal}/{n_assets} | Cond(LW)={cond_lw:.0f} vs Sample={cond_sample:.0f}',
        f'MC VaR@99%={VaR_mc/1e6:.2f}M, CVaR={CVaR_mc/1e6:.2f}M (10M portfolio)',
        f'BL instability: {std_bl.mean()*100:.2f}% vs Naive: {std_naive.mean()*100:.2f}%'
    ]
}

df_summary = pd.DataFrame(summary_data).set_index('Step')
print("=" * 100)
print("Portfolio Optimization — All 7 Steps (SET Data: 825 stocks, 2015-2025)")
print("=" * 100)
print(df_summary.to_string())
print("\nAll 7 steps complete!")